In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

In [12]:
df = pd.read_csv("titanic.csv")

In [13]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Создание X и Y

In [14]:
y = df["Survived"]
X = df[["Pclass", "Age", "Sex", "Fare", "SibSp", "Parch"]]
X.head()

,Pclass,Age,Sex,Fare,SibSp,Parch
0,3,22.0,male,7.2500,1,0
1,1,38.0,female,71.2833,1,0
2,3,26.0,female,7.9250,0,0
3,1,35.0,female,53.1000,1,0
4,3,35.0,male,8.0500,0,0


In [15]:
X.isnull().sum()

,0
Pclass,0
Age,177
Sex,0
Fare,0
SibSp,0
Parch,0


Обработка пропусков в Age

In [16]:
X.loc[:, "Age"] = X["Age"].fillna(X["Age"].median())
X.isnull().sum()

,0
Pclass,0
Age,0
Sex,0
Fare,0
SibSp,0
Parch,0


Преобразование категориального признака Sex в числовой

In [17]:
X["Sex"] = X["Sex"].map({
    "male": 0,
    "female": 1
})

X.head()

/tmp/ipykernel_2699/2466100129.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Sex"] = X["Sex"].map({


,Pclass,Age,Sex,Fare,SibSp,Parch
0,3,22.0,0,7.2500,1,0
1,1,38.0,1,71.2833,1,0
2,3,26.0,1,7.9250,0,0
3,1,35.0,1,53.1000,1,0
4,3,35.0,0,8.0500,0,0


Проверка подготовленных данных

In [18]:
X.info()
X.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Pclass  891 non-null    int64  
 1   Age     891 non-null    float64
 2   Sex     891 non-null    int64  
 3   Fare    891 non-null    float64
 4   SibSp   891 non-null    int64  
 5   Parch   891 non-null    int64  
dtypes: float64(2), int64(4)
memory usage: 41.9 KB


,0
Pclass,0
Age,0
Sex,0
Fare,0
SibSp,0
Parch,0


Разделить данные на обучающую и тестовую выборки

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Масштабирование признаков

In [20]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Вывод размеров полученных наборов

In [21]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (712, 6)
X_test: (179, 6)
y_train: (712,)
y_test: (179,)


In [24]:
# ПР6
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)

print("Первые предсказания:", y_pred_lr[:10])
print("Правильные ответы:  ", y_test.iloc[:10].to_list())

Первые предсказания: [0 0 0 0 1 0 1 0 0 0]
Правильные ответы:   [0, 0, 1, 0, 1, 1, 1, 0, 0, 0]


Зачем мы оставляем тестовую выборку отдельно? -
Чтобы честно оценить качество модели на тех данных, которые она не видела при обучении. Если проверять модель на обучающей выборке, оценка будет завышенной

Почему scaler.fit_transform() используется только для обучающей выборки?-
 Чтобы избежать утечки данных

 Почему в обучении используются X_train_scaled и y_train, а не тестовые данные?- Модель должна учиться находить закономерности только на тренировочных данных


Первые предсказания: [0 0 0 0 1 0 1 0 0 0]
Правильные ответы: [0, 0, 1, 0, 1, 1, 1, 0, 0, 0]

Совпавший ответ: Например, пассажир под индексом 1. Предсказание: 0
правильный ответ: 0. (Модель верно определила, что он не выжил)

Ошибочный ответ: Например, пассажир под индексом 2. Предсказание: 0, правильный ответ: 1. (Модель предсказала, что он погиб, хотя он выжил)

Что означает 0 и 1: 0 - пассажир не выжил, 1 - пассажир выжил.


In [26]:
print("Accuracy :", round(accuracy_score(y_test, y_pred_lr), 4))
print("Precision:", round(precision_score(y_test, y_pred_lr), 4))
print("Recall   :", round(recall_score(y_test, y_pred_lr), 4))
print("F1       :", round(f1_score(y_test, y_pred_lr), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy : 0.8045
Precision: 0.7656
Recall   : 0.7101
F1       : 0.7368
Confusion matrix:
 [[95 15]
 [20 49]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       110
           1       0.77      0.71      0.74        69

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [27]:
model_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
model_tree.fit(X_train, y_train)

y_pred_tree = model_tree.predict(X_test)

print("Accuracy дерева:", round(accuracy_score(y_test, y_pred_tree), 4))
print("Precision:", round(precision_score(y_test, y_pred_tree), 4))
print("Recall   :", round(recall_score(y_test, y_pred_tree), 4))
print("F1       :", round(f1_score(y_test, y_pred_tree), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_tree))

Accuracy дерева: 0.7821
Precision: 0.875
Recall   : 0.5072
F1       : 0.6422
Confusion matrix:
 [[105   5]
 [ 34  35]]


In [28]:
df_corr = df.copy()
df_corr["Sex"] = df_corr["Sex"].map({"male": 0, "female": 1})

print(df_corr[["Survived","Pclass","Age","Sex","Fare","SibSp","Parch"]]
      .corr()["Survived"].sort_values(ascending=False))

Survived    1.000000
Sex         0.543351
Fare        0.257307
Parch       0.081629
SibSp      -0.035322
Age        -0.077221
Pclass     -0.338481
Name: Survived, dtype: float64


Какой признак имел наиболее сильную положительную корреляцию с Survived?

Признак Sex (корреляция +0.543)

Какой признак имел наиболее сильную отрицательную корреляцию?

Признак Pclass (корреляция -0.338)
